In [1]:
import random
import torch
import os
import umap
import glob
import json
import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import (
    StratifiedKFold, cross_validate, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, make_scorer, confusion_matrix
)
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB

from scipy import sparse

import warnings
warnings.simplefilter("ignore", UserWarning)

In [2]:
N_SPLITS_CV = 5
N_JOBS = 1  # prioritizing no leakage
INPUT_DIR = '.'
OUTPUT_DIR = './machineLearningOutputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
# get models - broad coverage across linear, margin, instance-based, and nonlinear tree ensembles
def build_models(random_state):
    models = {
        # high numbers of iterations (max_iter) avoids convergence warnings in high-dimensional spaces
        "LogisticRegression": LogisticRegression(max_iter=20000, class_weight="balanced", random_state=random_state), # strong baseline
        "RidgeClassifier": RidgeClassifier(class_weight="balanced", random_state=random_state), # strong baseline
        "LinearSVC": LinearSVC(max_iter=20000, class_weight="balanced", random_state=random_state), # strong baseline
        "SVC-RBF": SVC(probability=False, random_state=random_state, class_weight="balanced"), # probability set to false saves on compute
        "SGD-Hinge": SGDClassifier(loss="hinge", max_iter=20000, random_state=random_state, class_weight="balanced"), # hinge loss is good for binary classification problems - fast on large data
        "KNN": KNeighborsClassifier(), # deterministic - sensitive to scaling (good to test with and without standard scaler)
        "DecisionTree": DecisionTreeClassifier(random_state=random_state, class_weight="balanced"), # don't need scaling and captures nonlinear splits
        "RandomForest": RandomForestClassifier(random_state=random_state, class_weight="balanced"), # don't need scaling and captures nonlinear splits
        "ExtraTrees": ExtraTreesClassifier(random_state=random_state, class_weight="balanced"), # don't need scaling and captures nonlinear splits
        "GradientBoosting": GradientBoostingClassifier(random_state=random_state), # don't need scaling and captures nonlinear splits
        "HistGB": HistGradientBoostingClassifier(random_state=random_state), # don't need scaling and captures nonlinear splits
        "AdaBoost": AdaBoostClassifier(random_state=random_state), # don't need scaling and captures nonlinear splits
        "GaussianNaiveBayes": GaussianNB(), # deterministic - interpretable
    }
    return models

# get parameter grid to test
def small_param_grid(name): # high-leverage parameter sweeps (runtime will be effected if we test everything)
    grids = {
        # lowered c values due to convergence issues
        "LogisticRegression": {"clf__C": [0.1, 0.5, 1.0]},
        "LinearSVC":          {"clf__C": [0.1, 0.5, 1.0]},
        "SVC-RBF":            {"clf__C": [0.1, 0.5, 1.0], "clf__gamma": ["scale", "auto"]},
        "KNN":                {"clf__n_neighbors": [3, 5, 11]},
        "RandomForest":       {"clf__n_estimators": [300, 600], "clf__max_depth": [None, 10, 20]},
        "ExtraTrees":         {"clf__n_estimators": [300, 600], "clf__max_depth": [None, 10, 20]},
        "HistGB":             {"clf__max_depth": [None, 6, 10]},
    }
    return grids.get(name, None)

# make pipeline with and without scaling
def preprocessor(scale):
    # median ensures it is robust to outliers
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        # scales data to benefit models such as SVM and KNN
        steps.append(("scaler", StandardScaler(with_mean=True)))
    else:
        steps.append(("scaler", FunctionTransformer(lambda X: X, feature_names_out="one-to-one")))
    return Pipeline(steps)


def build_scorers():
    return {
        "accuracy": "accuracy", # normal accuracy
        "balanced_accuracy": make_scorer(balanced_accuracy_score), # balanced accuracy
        "precision": make_scorer(precision_score, average="macro", zero_division=0),
        "recall": make_scorer(recall_score, average="macro", zero_division=0),
        "f1": make_scorer(f1_score, average="macro", zero_division=0)
    }

In [5]:
def evaluate_one(X_train, y_train, X_test, y_test, variant,
                 random_state=RANDOM_STATE, n_splits=N_SPLITS_CV, n_jobs=N_JOBS):

    inner_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    outer_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scorers = build_scorers()
    results = []

    for name, clf in build_models(random_state).items():
        # scaling decision
        X_tr, X_te = X_train, X_test

        # Convert sparse to dense for models that need dense arrays
        if sparse.issparse(X_tr):
            if name in ["HistGB", "GradientBoosting", "GaussianNaiveBayes"]:
                X_tr = X_tr.toarray()
                X_te = X_te.toarray()

        # Scaling decision
        if variant == "scaled":
            scaler = StandardScaler(with_mean=False if sparse.issparse(X_tr) else True)
        else:
            scaler = FunctionTransformer(lambda X: X, feature_names_out="one-to-one")

        pipe = Pipeline([
            ("scale", scaler),
            ("clf", clf)
        ])

        grid = small_param_grid(name)
        model_for_cv = GridSearchCV(pipe, grid, scoring=scorers, cv=inner_cv,
                                    n_jobs=n_jobs, refit="f1") if grid else pipe

        cv_out = cross_validate(model_for_cv, X_tr, y_train, cv=outer_cv,
                                scoring=scorers, return_train_score=False,
                                n_jobs=n_jobs, return_estimator=True)

        # Fit on full training
        model_for_cv.fit(X_tr, y_train)
        y_pred = model_for_cv.predict(X_te)

        # best estimator info
        if isinstance(model_for_cv, GridSearchCV):
            chosen = model_for_cv.best_estimator_
            tuned = model_for_cv.best_params_
        else:
            chosen = model_for_cv
            tuned = {}

        clf_params = chosen.named_steps["clf"].get_params()
        tuned_json = json.dumps(tuned, default=str)
        clf_json = json.dumps(clf_params, default=str)

        row = {
            "variant": variant,
            "model": name,
            "tuned_parameters": tuned_json,
            "clf_parameters": clf_json,
            **{f"cv_{k.replace('test_','')}_mean": float(np.mean(v))
               for k,v in cv_out.items() if k.startswith("test_")},
            **{f"cv_{k.replace('test_','')}_std": float(np.std(v))
               for k,v in cv_out.items() if k.startswith("test_")},
            "holdout_accuracy": accuracy_score(y_test, y_pred),
            "holdout_bal_acc": balanced_accuracy_score(y_test, y_pred),
            "holdout_precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
            "holdout_recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
            "holdout_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        }

        results.append(row)
        print(row)
    return pd.DataFrame(results)

In [6]:
def make_input_data(file_paths="./outputsTrain/*/mean_model_zero_shot_classification.csv"):
    # Get all CSV files.
    all_files = glob.glob(file_paths)  # change this to your folder path

    # Loop through files.
    all_dfs = []

    for f in all_files:
        df = pd.read_csv(f)
        df['category'] = df['doc_id'].apply(lambda x: x.split('_')[0])
        
        # Aggregate numeric factor columns by category
        factor_cols = [f"factor_{i}" for i in range(1, 7)]
        df_agg = df.groupby('category')[factor_cols].mean().reset_index()
        all_dfs.append(df_agg)

    # Concatenate all aggregated DataFrames
    dfs = pd.concat(all_dfs, ignore_index=True)

    dfs['category_codes'] = dfs['category'].astype('category').cat.codes
    # Return x and y arrays.
    return dfs[[f"factor_{i}" for i in range(1, 7)]].values, dfs['category_codes'].values

In [7]:
# Assess zero-shot performance. 
X_train, y_train = make_input_data(file_paths="./outputsTrain/*/mean_model_zero_shot_classification.csv")
X_test, y_test = make_input_data(file_paths="./outputsTest/*/mean_model_zero_shot_classification.csv")

all_results = []
for variant in ("raw", "scaled"):
    df_res = evaluate_one(X_train, y_train, X_test, y_test, variant)
    all_results.append(df_res)

# Concatenate all results and save.
final_results = pd.concat(all_results, ignore_index=True)
final_results.to_csv(f"./{OUTPUT_DIR}/zeroShotResults.csv", index=False)

for metric in ['holdout_accuracy', 'holdout_bal_acc', 'holdout_precision', 'holdout_recall', 'holdout_f1']:
    print(f"Highest: {final_results.sort_values(by=metric, ascending=False)[metric].tolist()[0]}")
    print(pl.from_pandas(final_results.sort_values(by=metric, ascending=False).head()))

{'variant': 'raw', 'model': 'LogisticRegression', 'tuned_parameters': '{"clf__C": 1.0}', 'clf_parameters': '{"C": 1.0, "class_weight": "balanced", "dual": false, "fit_intercept": true, "intercept_scaling": 1, "l1_ratio": 0.0, "max_iter": 20000, "n_jobs": null, "penalty": "deprecated", "random_state": 1618, "solver": "lbfgs", "tol": 0.0001, "verbose": 0, "warm_start": false}', 'cv_accuracy_mean': 0.923, 'cv_balanced_accuracy_mean': 0.9229999999999998, 'cv_precision_mean': 0.9263907626444239, 'cv_recall_mean': 0.9229999999999998, 'cv_f1_mean': 0.9225946684454313, 'cv_accuracy_std': 0.020880613017821063, 'cv_balanced_accuracy_std': 0.020880613017821112, 'cv_precision_std': 0.019936166041510675, 'cv_recall_std': 0.020880613017821112, 'cv_f1_std': 0.021138388018740236, 'holdout_accuracy': 0.936, 'holdout_bal_acc': 0.9359999999999999, 'holdout_precision': 0.9364491917125551, 'holdout_recall': 0.9359999999999999, 'holdout_f1': 0.9356933601120521}
{'variant': 'raw', 'model': 'RidgeClassifier',

In [8]:
# Assess MDA performance. 
X_train, y_train = make_input_data(file_paths="./outputsTrain/*/mda_dim_scores.csv")
X_test, y_test = make_input_data(file_paths="./outputsTest/*/mda_dim_scores.csv")

all_results = []
for variant in ("raw", "scaled"):
    df_res = evaluate_one(X_train, y_train, X_test, y_test, variant)
    all_results.append(df_res)

# Concatenate all results and save.
final_results = pd.concat(all_results, ignore_index=True)
final_results.to_csv(f"./{OUTPUT_DIR}/mdaResults.csv", index=False)

for metric in ['holdout_accuracy', 'holdout_bal_acc', 'holdout_precision', 'holdout_recall', 'holdout_f1']:
    print(f"Highest: {final_results.sort_values(by=metric, ascending=False)[metric].tolist()[0]}")
    print(pl.from_pandas(final_results.sort_values(by=metric, ascending=False).head()))

{'variant': 'raw', 'model': 'LogisticRegression', 'tuned_parameters': '{"clf__C": 1.0}', 'clf_parameters': '{"C": 1.0, "class_weight": "balanced", "dual": false, "fit_intercept": true, "intercept_scaling": 1, "l1_ratio": 0.0, "max_iter": 20000, "n_jobs": null, "penalty": "deprecated", "random_state": 1618, "solver": "lbfgs", "tol": 0.0001, "verbose": 0, "warm_start": false}', 'cv_accuracy_mean': 0.8619999999999999, 'cv_balanced_accuracy_mean': 0.8619999999999999, 'cv_precision_mean': 0.8641728060754847, 'cv_recall_mean': 0.8619999999999999, 'cv_f1_mean': 0.8607806171467693, 'cv_accuracy_std': 0.021587033144922923, 'cv_balanced_accuracy_std': 0.021587033144922847, 'cv_precision_std': 0.02078524931808598, 'cv_recall_std': 0.021587033144922847, 'cv_f1_std': 0.021434092741046896, 'holdout_accuracy': 0.864, 'holdout_bal_acc': 0.8640000000000001, 'holdout_precision': 0.860846564141078, 'holdout_recall': 0.8640000000000001, 'holdout_f1': 0.8610305796873756}
{'variant': 'raw', 'model': 'RidgeC